# Complementos de Investigação Operacional — capítulo 7: Gestão de stocks
### Exemplos em Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/applied-operations-research-course/blob/main/code/python/notebooks/07_inventory.ipynb)

Cada secção corre o exemplo `ex….py` do método e compara os resultados com os slides (no fim: *confere com os slides: sim*). Comece pela célula **Preparação**; no Colab, *Runtime → Run all* corre tudo.

Os slides usam vírgula decimal; aqui usa-se o ponto. Este caderno é gerado a partir dos ficheiros `ex….py` por `make_notebooks.py`.

*Examples for Chapter 7 of the course "Complements of Operational Research" (ISEL, J. F. A. Madeira). Code comments and printed messages are in Portuguese; the English slides are in [`notes/en/`](https://github.com/aguilarmadeira/applied-operations-research-course/tree/main/notes/en).*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("applied-operations-research-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/applied-operations-research-course
    RAIZ = os.path.abspath(os.path.join("applied-operations-research-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 7.1 Gestão de stocks: classificação ABC e quantidade económica de encomenda

Exemplo [`ex07_1_abc_eoq.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/07_inventory/07_1_abc_eoq/ex07_1_abc_eoq.py) · funções da UC: [`stocks.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/07_inventory/07_1_abc_eoq/stocks.py)

```text
Reproduz os exemplos do deck 7.1 (classificação ABC e QEE).

Fábrica de tintas, ABC: total 120 300 €/ano; A: P1, P2, P3 (78,8 %); B: P4, P5; C: P6–P10.
Dióxido de titânio (D = 12 000 kg/ano, Ce = 150 €, Cp = 0,2 x 4 = 0,80 €/kg/ano, prazo 2 semanas):
Q* = 2121 kg, 5,66 encomendas/ano, ciclo 9,2 semanas, encomenda + posse 1697,06 €, K = 49 697,06 €,
ponto de encomenda 461,5 kg.
Robustez: Q = 1500, 2000, 2500, 3000 -> 1800, 1700, 1720, 1800 €; com D = 14 400 e Q = 2121:
1866,76 € contra 1859,03 € do ótimo (+0,4 %).
Para resolver na aula: Exemplo 1 (ABC, total 62 498; A: A1, A8, A3; B: A2, A5) e
Exemplo 2 (transístores: Cp = 5,50, Q* = 988, K = 198 094, 4 semanas, Pe = 494, 13 encomendas).
```

In [2]:
from stocks import abc, qee, custo_qee


def mostra_abc(titulo, itens):
    tot, linhas = abc(itens)
    print("\n" + titulo + ": total %.2f" % tot)
    print("  artigo  valor/un.  quantidade      valor      %  acum.  classe")
    for k, v, q, val, p, c, cl in linhas:
        print("  %-6s %10.2f %11.0f %10.2f %6.1f %6.1f  %s" % (k, v, q, val, p, c, cl))
    classes = {c: [l[0] for l in linhas if l[6] == c] for c in "ABC"}
    print("  " + ";  ".join("%s: %s" % (c, ", ".join(classes[c])) for c in "ABC"))
    return tot, linhas, classes


ok = True
print("Complementos de IO — deck 7.1: classificação ABC e quantidade económica de encomenda")

# ------------------------------------------------------------ ABC da fábrica de tintas
tinta = {"P1": (4, 12000), "P2": (3.2, 9000), "P3": (0.9, 20000), "P4": (25, 400), "P5": (1.1, 6000),
         "P6": (12, 300), "P7": (0.15, 18000), "P8": (0.03, 40000), "P9": (0.05, 22000), "P10": (3, 100)}
tot, L, cl = mostra_abc("ABC da fábrica de tintas (€/ano)", tinta)
ok &= (abs(tot - 120300) < 1e-6 and cl["A"] == ["P1", "P2", "P3"] and cl["B"] == ["P4", "P5"]
       and cl["C"] == ["P6", "P7", "P8", "P9", "P10"] and abs(L[2][5] - 78.8) < 0.05
       and abs(L[4][5] - 92.6) < 0.05 and abs(L[5][5] - 95.6) < 0.05)

# ------------------------------------------------------------ QEE do dióxido de titânio
D, Ce, Ca = 12000, 150, 4
Cp = 0.2 * Ca
q = qee(D, Ce, Cp, Ca, TR=2 / 52)
g = q["Kenc"] + q["Kpos"]
print("\nQEE do dióxido de titânio (D = %d kg/ano, Ce = %d €, Cp = %.2f €/kg/ano, prazo 2 semanas)" % (D, Ce, Cp))
print("  Q* = %.2f kg;  encomendas por ano %.2f;  ciclo %.1f semanas" % (q["Q"], q["n"], q["T"] * 52))
print("  encomenda %.2f + posse %.2f = %.2f €;  custo total (com compra) %.2f €" % (q["Kenc"], q["Kpos"], g, q["K"]))
print("  ponto de encomenda %.1f kg" % q["Pe"])
ok &= (abs(q["Q"] - 2121.32) < 0.01 and abs(q["n"] - 5.66) < 0.005 and abs(q["T"] * 52 - 9.2) < 0.05
       and abs(g - 1697.06) < 0.005 and abs(q["K"] - 49697.06) < 0.005 and abs(q["Pe"] - 461.5) < 0.05)

# ------------------------------------------------------------ robustez
print("\nRobustez (encomenda + posse)")
esperado = {1500: 1800, 2000: 1700, 2500: 1720, 3000: 1800}
for Q in (1500, 2000, 2500, 3000):
    gQ = custo_qee(Q, D, Ce, Cp)
    print("  Q = %4d: %.2f €  (+%.2f%%)" % (Q, gQ, 100 * (gQ / g - 1)))
    ok &= abs(gQ - esperado[Q]) < 1e-6
g14 = custo_qee(q["Q"], 14400, Ce, Cp)
q14 = qee(14400, Ce, Cp)
o14 = q14["Kenc"] + q14["Kpos"]
print("  D = 14400 com Q = %.0f: %.2f €;  ótimo (Q = %.0f): %.2f €  (+%.1f%%)"
      % (q["Q"], g14, q14["Q"], o14, 100 * (g14 / o14 - 1)))
ok &= abs(g14 - 1866.76) < 0.005 and abs(o14 - 1859.03) < 0.005 and round(100 * (g14 / o14 - 1), 1) == 0.4

# ------------------------------------------------------------ Exemplo 1 (para resolver na aula)
ex1 = {"A1": (1, 22000), "A2": (12, 410), "A3": (4.25, 1468), "A4": (0.25, 3500), "A5": (2.25, 1600),
       "A6": (26, 10), "A7": (8.5, 124), "A8": (0.5, 40000), "A9": (1.25, 440), "A10": (0.12, 25000)}
tot1, L1, cl1 = mostra_abc("Exemplo 1 (ABC)", ex1)
ok &= (abs(tot1 - 62498) < 1e-6 and cl1["A"] == ["A1", "A8", "A3"] and cl1["B"] == ["A2", "A5"]
       and cl1["C"] == ["A10", "A7", "A4", "A9", "A6"] and abs(L1[5][5] - 95.6) < 0.05)

# ------------------------------------------------------------ Exemplo 2 (para resolver na aula)
Cp2 = 3.25 + 0.15 * 15
q2 = qee(12844, 209, Cp2, 15, TR=2 / 52)
print("\nExemplo 2 (caixa de transístores): Cp = 3.25 + 0.15 x 15 = %.2f" % Cp2)
print("  a) Q* = %.2f;  b) custo anual %.2f (compra %.0f + encomenda %.2f + posse %.2f)"
      % (q2["Q"], q2["K"], 12844 * 15, q2["Kenc"], q2["Kpos"]))
print("  c) ciclo %.2f semanas;  d) ponto de encomenda %.2f;  e) %.2f encomendas por ano"
      % (q2["T"] * 52, q2["Pe"], q2["n"]))
print("  f) posse / encomenda = %.4f" % (q2["Kpos"] / q2["Kenc"]))
ok &= (round(q2["Q"]) == 988 and round(q2["K"]) == 198094 and round(q2["T"] * 52) == 4
       and round(q2["Pe"]) == 494 and round(q2["n"]) == 13 and round(q2["Kenc"]) == 2717
       and abs(q2["Kpos"] / q2["Kenc"] - 1) < 1e-12)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 7.1: classificação ABC e quantidade económica de encomenda

ABC da fábrica de tintas (€/ano): total 120300.00
  artigo  valor/un.  quantidade      valor      %  acum.  classe
  P1           4.00       12000   48000.00   39.9   39.9  A
  P2           3.20        9000   28800.00   23.9   63.8  A
  P3           0.90       20000   18000.00   15.0   78.8  A
  P4          25.00         400   10000.00    8.3   87.1  B
  P5           1.10        6000    6600.00    5.5   92.6  B
  P6          12.00         300    3600.00    3.0   95.6  C
  P7           0.15       18000    2700.00    2.2   97.8  C
  P8           0.03       40000    1200.00    1.0   98.8  C
  P9           0.05       22000    1100.00    0.9   99.8  C
  P10          3.00         100     300.00    0.2  100.0  C
  A: P1, P2, P3;  B: P4, P5;  C: P6, P7, P8, P9, P10

QEE do dióxido de titânio (D = 12000 kg/ano, Ce = 150 €, Cp = 0.80 €/kg/ano, prazo 2 semanas)
  Q* = 2121.32 kg;  encomendas por ano 5.66;  ciclo

## 7.2 Gestão de stocks: rutura planeada e descontos de quantidade

Exemplo [`ex07_2_shortages_discounts.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/07_inventory/07_2_shortages_discounts/ex07_2_shortages_discounts.py) · funções da UC: [`rutura_descontos.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/07_inventory/07_2_shortages_discounts/rutura_descontos.py), [`stocks.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/07_inventory/07_1_abc_eoq/stocks.py)

```text
Reproduz os exemplos do deck 7.2 (rutura planeada e descontos de quantidade).

Tinta premium com rutura (D = 1200 latas/ano, Ca = 50 €, Ce = 80 €, Cp = 10 €, Cr = 30 €, prazo 1 semana):
rho = 0,75, Q* = 160, S* = 120, rutura máxima 40, ciclo 1,6 meses (6,9 semanas), 5,2 / 1,7 semanas,
Pe = 23,1 - 40 = -16,9, 7,5 encomendas/ano; encomenda 600 + posse 450 + rutura 150 = 1200 €
contra 1385,64 € sem rutura (poupa 185,64 €, 13 %); custo total 61 200 €.
Descontos no dióxido de titânio: 49 697,06, 48 324, 48 180 € -> encomendar 6000 kg
(poupa 1517 € face à QEE sem desconto; só 144 € abaixo de 3000 kg).
Para resolver na aula: Exemplo 3 (loja AAA -> 400 unidades, 3453,19 €) e
Exemplo 4 (componentes -> 6000, 1 419 975 u.m., 2,5 encomendas/ano, ciclo 0,4 anos).
```

In [3]:
from rutura_descontos import rutura, descontos
from stocks import qee


def mostra_descontos(titulo, D, Ce, I, escaloes):
    linhas, melhor = descontos(D, Ce, I, escaloes)
    print("\n" + titulo + ": D = %g, Ce = %g, posse %g%% do preço" % (D, Ce, 100 * I))
    print("  escalão        preço      QEE  Q usado         compra  encomenda      posse          total")
    for l in linhas:
        esc = "%g+" % l["lo"] if l["hi"] >= 1e9 else "%g-%g" % (l["lo"], l["hi"])
        Cp = I * l["Ca"]
        print("  %-11s %8.4g %8.1f %8.1f %14.2f %10.2f %10.2f %14.2f"
              % (esc, l["Ca"], l["Q"], l["Qf"], D * l["Ca"], D * Ce / l["Qf"], Cp * l["Qf"] / 2, l["K"]))
    print("  -> encomendar %.0f: custo %.2f;  %.2f encomendas por ano;  ciclo T = Q/D = %.4f"
          % (melhor["Qf"], melhor["K"], D / melhor["Qf"], melhor["Qf"] / D))
    return linhas, melhor


ok = True
print("Complementos de IO — deck 7.2: rutura planeada e descontos de quantidade")

# ------------------------------------------------------------ tinta premium com rutura
r = rutura(1200, 80, 10, 30, 50, TR=1 / 52)
print("\nTinta premium com rutura (D = 1200 latas/ano, Ce = 80, Cp = 10, Cr = 30, Ca = 50, prazo 1 semana)")
print("  rho = %.2f;  Q* = %.2f;  S* = %.2f;  rutura máxima %.2f" % (r["rho"], r["Q"], r["S"], r["R"]))
print("  ciclo %.2f meses (%.2f semanas);  com stock %.2f semanas, em rutura %.2f semanas"
      % (r["T"] * 12, r["T"] * 52, r["T1"] * 52, r["T2"] * 52))
print("  ponto de encomenda %.2f - %.2f = %.2f;  %.2f encomendas por ano"
      % (1200 / 52, r["R"], r["Pe"], r["n"]))
s = qee(1200, 80, 10)
gs = s["Kenc"] + s["Kpos"]
gr = r["Kenc"] + r["Kpos"] + r["Krut"]
print("  €/ano         com rutura  sem rutura")
print("  encomenda     %10.2f  %10.2f" % (r["Kenc"], s["Kenc"]))
print("  posse         %10.2f  %10.2f" % (r["Kpos"], s["Kpos"]))
print("  rutura        %10.2f" % r["Krut"])
print("  soma          %10.2f  %10.2f" % (gr, gs))
print("  poupança %.2f €/ano (%.1f%%);  compra %.0f;  custo total %.2f" % (gs - gr, 100 * (1 - gr / gs), r["Kaq"], r["K"]))
ok &= (abs(r["rho"] - 0.75) < 1e-12 and abs(r["Q"] - 160) < 1e-9 and abs(r["S"] - 120) < 1e-9
       and abs(r["R"] - 40) < 1e-9 and abs(r["T"] * 12 - 1.6) < 1e-9 and round(r["T"] * 52, 1) == 6.9
       and round(r["T1"] * 52, 1) == 5.2 and round(r["T2"] * 52, 1) == 1.7 and round(r["Pe"], 1) == -16.9
       and abs(r["n"] - 7.5) < 1e-9 and abs(r["Kenc"] - 600) < 1e-9 and abs(r["Kpos"] - 450) < 1e-9
       and abs(r["Krut"] - 150) < 1e-9 and abs(gs - 1385.64) < 0.005 and abs(gs - gr - 185.64) < 0.005
       and round(100 * (1 - gr / gs)) == 13 and abs(r["K"] - 61200) < 1e-6)

# ------------------------------------------------------------ descontos no dióxido de titânio
L, b = mostra_descontos("Descontos no dióxido de titânio", 12000, 150, 0.2,
                        [(0, 2999, 4), (3000, 5999, 3.88), (6000, 1e9, 3.8)])
print("  poupa %.2f €/ano face à QEE sem desconto;  %.2f abaixo de 3000 kg" % (L[0]["K"] - b["K"], L[1]["K"] - b["K"]))
ok &= ([round(l["Q"]) for l in L] == [2121, 2154, 2176] and [round(l["Qf"]) for l in L] == [2121, 3000, 6000]
       and abs(L[0]["K"] - 49697.06) < 0.005 and abs(L[1]["K"] - 48324) < 1e-6 and abs(L[2]["K"] - 48180) < 1e-6
       and b["Qf"] == 6000 and round(L[0]["K"] - b["K"]) == 1517 and abs(L[1]["K"] - b["K"] - 144) < 1e-6)

# ------------------------------------------------------------ Exemplo 3 (para resolver na aula)
L3, b3 = mostra_descontos("Exemplo 3, loja AAA (D = 21 x 52 por ano)", 21 * 52, 20, 0.25,
                          [(0, 399, 3.2), (400, 899, 3.2 * 0.93), (900, 1999, 3.2 * 0.9), (2000, 1e9, 3.2 * 0.85)])
print("  ciclo %.1f semanas" % (52 * b3["Qf"] / (21 * 52)))
ok &= ([round(l["K"], 2) for l in L3] == [3681.33, 3453.19, 3493.23, 3661.16] and b3["Qf"] == 400
       and [round(l["Q"], 1) for l in L3] == [233.7, 242.3, 246.3, 253.4]
       and round(21 * 52 / b3["Qf"], 2) == 2.73 and round(52 * b3["Qf"] / (21 * 52), 1) == 19.0)

# ------------------------------------------------------------ Exemplo 4 (para resolver na aula)
L4, b4 = mostra_descontos("Exemplo 4, componentes importados", 15000, 150, 0.2,
                          [(0, 999, 100), (1000, 2999, 95), (3000, 5999, 94), (6000, 8999, 91), (9000, 1e9, 90)])
print("  ciclo %.1f anos = %.1f meses" % (b4["Qf"] / 15000, 12 * b4["Qf"] / 15000))
ok &= ([round(l["K"], 2) for l in L4] == [1509486.83, 1436750, 1438950, 1419975, 1431250] and b4["Qf"] == 6000
       and [round(l["Q"], 1) for l in L4] == [474.3, 486.7, 489.2, 497.2, 500.0]
       and abs(15000 / b4["Qf"] - 2.5) < 1e-12 and abs(b4["Qf"] / 15000 - 0.4) < 1e-12)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 7.2: rutura planeada e descontos de quantidade

Tinta premium com rutura (D = 1200 latas/ano, Ce = 80, Cp = 10, Cr = 30, Ca = 50, prazo 1 semana)
  rho = 0.75;  Q* = 160.00;  S* = 120.00;  rutura máxima 40.00
  ciclo 1.60 meses (6.93 semanas);  com stock 5.20 semanas, em rutura 1.73 semanas
  ponto de encomenda 23.08 - 40.00 = -16.92;  7.50 encomendas por ano
  €/ano         com rutura  sem rutura
  encomenda         600.00      692.82
  posse             450.00      692.82
  rutura            150.00
  soma             1200.00     1385.64
  poupança 185.64 €/ano (13.4%);  compra 60000;  custo total 61200.00

Descontos no dióxido de titânio: D = 12000, Ce = 150, posse 20% do preço
  escalão        preço      QEE  Q usado         compra  encomenda      posse          total
  0-2999             4   2121.3   2121.3       48000.00     848.53     848.53       49697.06
  3000-5999       3.88   2153.9   3000.0       46560.00     600.00    1164.00       48324.00
  6000

## 7.3 Gestão de stocks: produção e consumo simultâneos

Exemplo [`ex07_3_production.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/07_inventory/07_3_production/ex07_3_production.py) · funções da UC: [`producao_consumo.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/07_inventory/07_3_production/producao_consumo.py), [`stocks.py`](https://github.com/aguilarmadeira/applied-operations-research-course/blob/main/code/python/07_inventory/07_1_abc_eoq/stocks.py)

```text
Reproduz os exemplos do deck 7.3 (produção e consumo simultâneos).

Tinta de interior (D = 24 000 L/ano, P = 60 000 L/ano, Ce = 900 €, custo 2 €/L, Cp = 0,25 x 2 = 0,50 €/L/ano):
Q* = 9295 x sqrt(60/36) = 12 000 L, stock máximo 7200 L, 2,4 meses a produzir, ciclo 6 meses, 2 lotes/ano,
preparação + posse = 1800 + 1800 = 3600 €, custo total 51 600 €.
Comparação: lote de uma vez (QEE) 9295 L e 4648 €/ano; o lote ótimo aumenta 29 %; linha ocupada 40 %.
Para resolver na aula (ano de 300 dias, preparação de 5 dias):
Exemplo 5 (resina): Q* = 1400 t, Smax = 350 t, T = 28 dias, Tp = 21 dias, lançamento a 250 t, K = 302 100 000;
Exemplo 6: Q* = 3487,1 t, Smax = 697,4 t, T = 52,3 dias, Tp = 41,85 dias, lançamento a 333,3 t, K = 100 871 779,8.
```

In [4]:
from math import sqrt

from producao_consumo import producao
from stocks import qee

ok = True
print("Complementos de IO — deck 7.3: produção e consumo simultâneos")

# ------------------------------------------------------------ tinta de interior
D, P, Ce, Ca = 24000, 60000, 900, 2
Cp = 0.25 * Ca
p = producao(D, P, Ce, Cp, Ca)
qs = qee(D, Ce, Cp)
gs = qs["Kenc"] + qs["Kpos"]
print("\nTinta de interior (D = %d L/ano, P = %d L/ano, Ce = %d, Cp = %.2f)" % (D, P, Ce, Cp))
print("  Q* = %.2f x %.4f = %.2f L;  stock máximo %.2f L" % (qs["Q"], sqrt(P / (P - D)), p["Q"], p["Smax"]))
print("  tempo a produzir %.2f meses;  ciclo %.2f meses;  %.2f lotes por ano" % (p["Tp"] * 12, p["T"] * 12, p["n"]))
print("  preparação %.2f + posse %.2f = %.2f €;  custo total (com produção) %.2f €"
      % (p["Kenc"], p["Kpos"], p["Kenc"] + p["Kpos"], p["K"]))
print("  lote de uma vez (QEE): Q = %.2f L, custo de gestão %.2f €;  lote ótimo +%.1f%%;  linha ocupada %.0f%%"
      % (qs["Q"], gs, 100 * (p["Q"] / qs["Q"] - 1), 100 * D / P))
ok &= (round(qs["Q"]) == 9295 and abs(p["Q"] - 12000) < 1e-6 and abs(p["Smax"] - 7200) < 1e-6
       and abs(p["Tp"] * 12 - 2.4) < 1e-9 and abs(p["T"] * 12 - 6) < 1e-9 and abs(p["n"] - 2) < 1e-9
       and abs(p["Kenc"] - 1800) < 1e-6 and abs(p["Kpos"] - 1800) < 1e-6 and abs(p["K"] - 51600) < 1e-6
       and round(gs) == 4648 and round(100 * (p["Q"] / qs["Q"] - 1)) == 29 and round(sqrt(60 / 36), 2) == 1.29)

# ------------------------------------------------------------ Exemplos 5 e 6 (para resolver na aula)
res = {}
for nome, P_, D_, Ce_, Ca_, I in (("Exemplo 5 (resina)", 20000, 15000, 98000, 20000, 0.3),
                                  ("Exemplo 6", 25000, 20000, 76000, 5000, 0.25)):
    Cp_ = I * Ca_
    r = producao(D_, P_, Ce_, Cp_, Ca_, Tprep=5 / 300)
    res[nome] = r
    print("\n%s: D = %d t/ano, P = %d t/ano, Ce = %d, Cp = %g x %d = %g (ano de 300 dias)"
          % (nome, D_, P_, Ce_, I, Ca_, Cp_))
    print("  Q* = %.2f x %.4f = %.2f t;  stock máximo %.2f t"
          % (sqrt(2 * D_ * Ce_ / Cp_), sqrt(P_ / (P_ - D_)), r["Q"], r["Smax"]))
    print("  ciclo %.2f dias;  a produzir %.2f dias;  sem produção %.2f dias;  %.2f lotes por ano"
          % (r["T"] * 300, r["Tp"] * 300, (r["T"] - r["Tp"]) * 300, r["n"]))
    print("  ponto de lançamento da produção %.2f t (preparação de 5 dias)" % r["Pl"])
    print("  produção %.2f + preparação %.2f + posse %.2f = %.2f por ano"
          % (D_ * Ca_, r["Kenc"], r["Kpos"], r["K"]))
r5, r6 = res["Exemplo 5 (resina)"], res["Exemplo 6"]
ok &= (abs(r5["Q"] - 1400) < 1e-6 and abs(r5["Smax"] - 350) < 1e-6 and abs(r5["T"] * 300 - 28) < 1e-9
       and abs(r5["Tp"] * 300 - 21) < 1e-9 and abs(r5["Pl"] - 250) < 1e-9 and abs(r5["K"] - 302100000) < 1e-3
       and round(r5["n"], 1) == 10.7)
ok &= (round(r6["Q"], 1) == 3487.1 and round(r6["Smax"], 1) == 697.4 and round(r6["T"] * 300, 1) == 52.3
       and abs(r6["Tp"] * 300 - 41.85) < 0.005 and round(r6["Pl"], 1) == 333.3 and round(r6["K"], 1) == 100871779.8
       and round(r6["n"], 1) == 5.7 and round((r6["T"] - r6["Tp"]) * 300, 1) == 10.5)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Complementos de IO — deck 7.3: produção e consumo simultâneos

Tinta de interior (D = 24000 L/ano, P = 60000 L/ano, Ce = 900, Cp = 0.50)
  Q* = 9295.16 x 1.2910 = 12000.00 L;  stock máximo 7200.00 L
  tempo a produzir 2.40 meses;  ciclo 6.00 meses;  2.00 lotes por ano
  preparação 1800.00 + posse 1800.00 = 3600.00 €;  custo total (com produção) 51600.00 €
  lote de uma vez (QEE): Q = 9295.16 L, custo de gestão 4647.58 €;  lote ótimo +29.1%;  linha ocupada 40%

Exemplo 5 (resina): D = 15000 t/ano, P = 20000 t/ano, Ce = 98000, Cp = 0.3 x 20000 = 6000 (ano de 300 dias)
  Q* = 700.00 x 2.0000 = 1400.00 t;  stock máximo 350.00 t
  ciclo 28.00 dias;  a produzir 21.00 dias;  sem produção 7.00 dias;  10.71 lotes por ano
  ponto de lançamento da produção 250.00 t (preparação de 5 dias)
  produção 300000000.00 + preparação 1050000.00 + posse 1050000.00 = 302100000.00 por ano

Exemplo 6: D = 20000 t/ano, P = 25000 t/ano, Ce = 76000, Cp = 0.25 x 5000 = 1250 (ano de 300 dias)
  Q* = 1559.49 x 2.236